In [23]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [24]:
import networkx as nx
import os

def merge_gexf_files(file_paths, output_path):
    """
    Merges multiple GEXF files into one.
    Nodes with the same ID will have their attributes merged.
    """
    merged_graph = nx.Graph()

    for path in file_paths:
        if not os.path.exists(path):
            print(f"Warning: File not found: {path}")
            continue

        print(f"Processing: {path}")
        # Load the graph
        current_graph = nx.read_gexf(path)

        # Merge nodes and attributes
        for node, data in current_graph.nodes(data=True):
            if merged_graph.has_node(node):
                # Update existing node with new attributes
                merged_graph.nodes[node].update(data)
            else:
                merged_graph.add_node(node, **data)

        # Merge edges
        for u, v, data in current_graph.edges(data=True):
            if not merged_graph.has_edge(u, v):
                merged_graph.add_edge(u, v, **data)

    # Save the result (creating output directory if needed)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    nx.write_gexf(merged_graph, output_path)
    print(f"Successfully created merged graph at: {output_path}")
    return merged_graph

# ── REGISTRO DE DIMENSIONES ─────────────────────────────────────────────
# Para añadir una nueva dimensión (postura, vestimenta...):
#   1. Crea pipeline/04_DIMENSION.ipynb que genere output/DIMENSION/grafo_DIMENSION.gexf
#   2. Añade la ruta aquí. El paso 3 fusiona todo automáticamente.
# ─────────────────────────────────────────────────────────────────────────
files_to_merge = [
    '/content/drive/MyDrive/TFM-Sara/output/transcripciones/knowledge_graph_palabras.gexf',   # dim: transcripcion
    '/content/drive/MyDrive/TFM-Sara/output/predicciones_año/grafo_años.gexf',                # dim: año
    # '/content/drive/MyDrive/TFM-Sara/output/postura/grafo_postura.gexf',                    # dim: postura  (futuro)
    '/content/drive/MyDrive/TFM-Sara/output/vestimenta/grafo_vestimenta.gexf',              # dim: vestimenta
]

output_file = '/content/drive/MyDrive/TFM-Sara/output/grafo_final/grafo_multidimensional.gexf'

# Execute merge
full_graph = merge_gexf_files(files_to_merge, output_file)

# ── Post-merge: normalizar nodos imagen ──────────────────────────────────
# Tras la fusión los nodos imagen pueden tener colores/atributos inconsistentes
# (azul claro del paso 1 vs rojo del paso 2). Los unificamos aquí.
from collections import Counter
dimension_counts = Counter()

for node, data in full_graph.nodes(data=True):
    dim = data.get('dimension', 'desconocida')
    dimension_counts[dim] += 1
    # Normalizar nodos imagen: color y group consistentes
    if data.get('group') == 1 or data.get('color') in ('lightblue', '#FF0000', '#4A90D9'):
        full_graph.nodes[node]['color']     = '#4A90D9'
        full_graph.nodes[node]['group']     = 1
        full_graph.nodes[node]['dimension'] = 'imagen'
        full_graph.nodes[node]['size']      = data.get('size', 25)

# Re-guardar con nodos normalizados
nx.write_gexf(full_graph, output_file)

print(f"Nodos por dimensión:")
for dim, count in sorted(dimension_counts.items()):
    print(f"  {dim:20s}: {count}")
from collections import Counter as _Counter2
rel_counts = _Counter2(d.get("relation", "sin_tipo") for _, _, d in full_graph.edges(data=True))
print(f"Total nodos  : {full_graph.number_of_nodes()}")
print(f"Total aristas: {full_graph.number_of_edges()}")
print("Aristas por tipo:")
for rel, cnt in sorted(rel_counts.items()):
    print(f"  {rel:25s}: {cnt}")

Successfully created merged graph at: /content/drive/MyDrive/TFM-Sara/output/grafo_final/grafo_multidimensional.gexf
Nodos por dimensión:
Total nodos  : 0
Total aristas: 0
Aristas por tipo:


In [25]:
import subprocess
subprocess.run(['umount', '-f', '/content/drive'], capture_output=True)


CompletedProcess(args=['umount', '-f', '/content/drive'], returncode=0, stdout=b'', stderr=b'')

In [26]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

ValueError: Mountpoint must not already contain files

In [29]:
# ── Enriquecimiento, comunidades y métricas ──────────────────────────────
import json
import pandas as pd

# 1. Enriquecer nodos foto con support_data/input.csv
SUPPORT_CSV = '/content/drive/MyDrive/TFM-Sara/support_data/input.csv'
if os.path.exists(SUPPORT_CSV):
    df_support = pd.read_csv(SUPPORT_CSV)
    support_lookup = {}
    for _, row in df_support.iterrows():
        fn = str(row.get('filename', '')).strip()
        if fn:
            support_lookup[fn] = {
                'caption':     str(row['caption'])    if pd.notna(row.get('caption'))     else '',
                'nom_fons':    str(row['nom_fons'])   if pd.notna(row.get('nom_fons'))    else '',
                'toponims':    str(row['toponims'])   if pd.notna(row.get('toponims'))    else '',
                'noms_propis': str(row['noms_propis'])if pd.notna(row.get('noms_propis')) else '',
                'year_csv':    int(row['year'])       if pd.notna(row.get('year'))        else None,
            }
    n_enriched = 0
    for node, data in full_graph.nodes(data=True):
        if data.get('dimension') == 'imagen' and node in support_lookup:
            for k, v in support_lookup[node].items():
                if v:
                    full_graph.nodes[node][k] = v
            n_enriched += 1
    print(f'✅ Nodos foto enriquecidos con CSV: {n_enriched}')
else:
    print(f'⚠️  {SUPPORT_CSV} no encontrado')

# 2. Aristas de similitud visual entre fotos de entrada
PRED_JSON = '/content/drive/MyDrive/TFM-Sara/output/predicciones_año/grafo_predicciones.json'
if os.path.exists(PRED_JSON):
    with open(PRED_JSON, 'r', encoding='utf-8') as f:
        predicciones = json.load(f)
    foto_vecinos = {foto: set(c['target'] for c in info.get('conexiones', []))
                    for foto, info in predicciones.items()}
    fotos = list(foto_vecinos.keys())
    n_sim = 0
    for i in range(len(fotos)):
        for j in range(i + 1, len(fotos)):
            shared = foto_vecinos[fotos[i]] & foto_vecinos[fotos[j]]
            if shared and full_graph.has_node(fotos[i]) and full_graph.has_node(fotos[j]):
                if not full_graph.has_edge(fotos[i], fotos[j]):
                    full_graph.add_edge(fotos[i], fotos[j],
                                        relation='similar_visualmente',
                                        weight=float(len(shared)),
                                        dimension='visual')
                    n_sim += 1
    print(f'✅ Aristas similitud visual: {n_sim}')

# 3. Aristas palabra → fecha (ponderadas por coocurrencia)
year_word_counts = {}
for u, v, data in list(full_graph.edges(data=True)):
    if data.get('relation') == 'contiene_palabra':
        foto = u if full_graph.nodes[u].get('dimension') == 'imagen' else v
        word = v if full_graph.nodes[u].get('dimension') == 'imagen' else u
        for nbr in full_graph.neighbors(foto):
            if full_graph.nodes[nbr].get('dimension') == 'año':
                key = (nbr, word)
                year_word_counts[key] = year_word_counts.get(key, 0) + 1

n_pw = 0
for (year_node, word_node), count in year_word_counts.items():
    if not full_graph.has_edge(word_node, year_node):
        full_graph.add_edge(word_node, year_node,
                            relation='aparece_en_año',
                            weight=float(count),
                            dimension='transcripcion_año')
        n_pw += 1
print(f'✅ Aristas palabra→fecha: {n_pw}')



# 3b. Aristas vestimenta -> fecha (ponderadas por coocurrencia)
vest_year_counts = {}
for u, v, data in list(full_graph.edges(data=True)):
    if data.get('relation') == 'lleva_puesto':
        foto = u if full_graph.nodes[u].get('dimension') == 'imagen' else v
        vest = v if full_graph.nodes[u].get('dimension') == 'imagen' else u
        for nbr in full_graph.neighbors(foto):
            if full_graph.nodes[nbr].get('dimension') == 'año':
                key = (nbr, vest)
                vest_year_counts[key] = vest_year_counts.get(key, 0) + 1

n_vy = 0
for (year_node, vest_node), count in vest_year_counts.items():
    if not full_graph.has_edge(vest_node, year_node):
        full_graph.add_edge(vest_node, year_node,
                            relation='vestimenta_en_año',
                            weight=float(count),
                            dimension='vestimenta_año')
        n_vy += 1
print(f'✅ Aristas vestimenta->fecha: {n_vy}')
# 4. Detección de comunidades (Louvain)
try:
    import community as community_louvain
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-louvain', '-q'])
    import community as community_louvain

partition = community_louvain.best_partition(full_graph, random_state=42)
COMMUNITY_PALETTE = [
    '#E63946','#457B9D','#2A9D8F','#E9C46A','#F4A261',
    '#8338EC','#3A86FF','#FB5607','#06D6A0','#118AB2',
    '#FFD166','#073B4C','#EF476F','#A8DADC','#6D6875',
    '#B5838D','#E07A5F','#3D405B','#81B29A','#F2CC8F',
]
for node, comm_id in partition.items():
    full_graph.nodes[node]['community'] = comm_id
    full_graph.nodes[node]['community_color'] = COMMUNITY_PALETTE[comm_id % len(COMMUNITY_PALETTE)]
from collections import Counter as _CC
comm_sizes = _CC(partition.values())
n_comm = max(partition.values()) + 1
print(f'✅ Comunidades detectadas: {n_comm}')
for cid, size in comm_sizes.most_common(5):
    print(f'   Comunidad {cid}: {size} nodos  {COMMUNITY_PALETTE[cid % len(COMMUNITY_PALETTE)]}')

# 5. Métricas de grafo
deg_c  = nx.degree_centrality(full_graph)
betw_c = nx.betweenness_centrality(full_graph, normalized=True)
clust  = nx.clustering(full_graph)
for node in full_graph.nodes():
    full_graph.nodes[node]['degree']       = full_graph.degree(node)
    full_graph.nodes[node]['deg_centrality']= round(deg_c[node], 6)
    full_graph.nodes[node]['betweenness']  = round(betw_c[node], 6)
    full_graph.nodes[node]['clustering']   = round(clust[node], 6)
top5 = sorted(betw_c.items(), key=lambda x: -x[1])[:5]
print('✅ Top 5 por betweenness centrality:')
for nid, val in top5:
    print(f'   {nid[:45]:45s} [{full_graph.nodes[nid].get("dimension","?"):12s}] {val:.4f}')

# 6. Re-guardar grafo enriquecido
nx.write_gexf(full_graph, output_file)
print(f'\n✅ Grafo guardado: {full_graph.number_of_nodes()} nodos · {full_graph.number_of_edges()} aristas')

# 7. Exportar nodos.csv y aristas.csv
_output_dir = '/content/drive/MyDrive/TFM-Sara/output/grafo_final'
os.makedirs(_output_dir, exist_ok=True)
rows_n = [{'id': nid, **data} for nid, data in full_graph.nodes(data=True)]
rows_e = [{'source': u, 'target': v, **data} for u, v, data in full_graph.edges(data=True)]
pd.DataFrame(rows_n).to_csv(f'{_output_dir}/nodos.csv',   index=False, encoding='utf-8')
pd.DataFrame(rows_e).to_csv(f'{_output_dir}/aristas.csv', index=False, encoding='utf-8')
print(f'✅ nodos.csv ({len(rows_n)}) · aristas.csv ({len(rows_e)})')
# 

⚠️  /content/drive/MyDrive/TFM-Sara/support_data/input.csv no encontrado
✅ Aristas palabra→fecha: 0


AttributeError: module 'community' has no attribute 'best_partition'

In [30]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
# Limpiar conflicto de paquetes
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'community', '-y', '-q'],
               capture_output=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-louvain', '-q'])

import re, os, json
import networkx as nx
import pandas as pd
from collections import Counter
from io import StringIO
import community as community_louvain
print(f"✅ Louvain OK: {hasattr(community_louvain, 'best_partition')}")

ValueError: Mountpoint must not already contain files

In [31]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', 'community', '-y', '-q'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'python-louvain', '-q'])
print("✅ Listo")

✅ Listo


In [ ]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive/TFM-Sara'):
    for f in files:
        print(os.path.join(root, f))

/content/drive/MyDrive/TFM-Sara/output/grafo_final/grafo_multidimensional.gexf


In [ ]:
from collections import Counter
dims = Counter(data.get('dimension', 'SIN_DIMENSION')
               for _, data in full_graph.nodes(data=True))
print(dims)

Counter()


In [ ]:
# ── Inyectar atributos viz: en el GEXF ─────────────────────────────────────
# El namespace viz: (GEXF 1.2) embebe color, tamaño y forma directamente
# en el fichero. Gephi y Gephi Lite los aplican al abrir sin configuración.
import xml.etree.ElementTree as ET

GEXF_NS  = 'http://www.gexf.net/1.2draft'
VIZ_NS   = 'http://www.gexf.net/1.2draft/viz'
ET.register_namespace('',    GEXF_NS)
ET.register_namespace('viz', VIZ_NS)

# Paleta de colores por dimensión (r, g, b) y forma
DIM_STYLE = {
    'imagen':       {'r': 74,  'g': 144, 'b': 217, 'shape': 'disc'},
    'transcripcion':{'r': 92,  'g': 184, 'b': 92,  'shape': 'disc'},
    'año':          {'r': 26,  'g': 58,  'b': 92,  'shape': 'square'},
    'postura':      {'r': 230, 'g': 126, 'b': 34,  'shape': 'diamond'},
    'vestimenta':   {'r': 155, 'g': 89,  'b': 182, 'shape': 'triangle'},
}

def _hex_to_rgb(hex_color):
    h = hex_color.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def inject_viz_namespace(gexf_path):
    tree = ET.parse(gexf_path)
    root = tree.getroot()

    # Añadir declaración del namespace viz al elemento raíz
    root.set('xmlns:viz', VIZ_NS)

    graph_el = root.find(f'{{{GEXF_NS}}}graph')
    nodes_el  = graph_el.find(f'{{{GEXF_NS}}}nodes')

    for node_el in nodes_el.findall(f'{{{GEXF_NS}}}node'):
        node_id = node_el.get('id')
        if node_id not in full_graph.nodes:
            continue

        data  = full_graph.nodes[node_id]
        dim   = data.get('dimension', 'imagen')
        style = DIM_STYLE.get(dim, DIM_STYLE['imagen'])
        size  = float(data.get('size', 25))

        # viz:color — nodos imagen: color por comunidad; resto: color por dimensión
        comm_hex = data.get('community_color', None)
        if dim == 'imagen' and comm_hex:
            r, g, b = _hex_to_rgb(comm_hex)
        else:
            r, g, b = style['r'], style['g'], style['b']
        color_el = ET.SubElement(node_el, f'{{{VIZ_NS}}}color')
        color_el.set('r', str(r))
        color_el.set('g', str(g))
        color_el.set('b', str(b))
        color_el.set('a', '255')

        # viz:size
        size_el = ET.SubElement(node_el, f'{{{VIZ_NS}}}size')
        size_el.set('value', f'{size:.1f}')

        # viz:shape
        shape_el = ET.SubElement(node_el, f'{{{VIZ_NS}}}shape')
        shape_el.set('value', style['shape'])

    tree.write(gexf_path, encoding='utf-8', xml_declaration=True)
    print(f'✅ viz: namespace inyectado en {os.path.basename(gexf_path)}')
    print('   Gephi / Gephi Lite abrirán el grafo ya con colores y formas aplicados.')

inject_viz_namespace(output_file)